# Lab: Generative AI for Data Science

In this lab I'm working through three scenarios where I prompt an AI (used Claude) to help generate starter code, then I further refine my prompts to generate more complete code.

The general workflow I followed for each scenario:

1. Write a prompt
2. Save the generated code
3. Follow-up prompts to refine the code
4. Save the generated code
5. Answer the discussion questions

The notes below document what I gave the AI, what came back

---

## Scenario 1: Retail Inventory Analysis

### Setup

The store manager wants to know which items are moving slowly and which ones are likely to run out before the next order arrives. The data has `product_id`, `category`, `stock_level`, `last_restock_date`, `sales_last_30_days`, `supplier_lead_time`, and `unit_cost`.

### My Prompt

```
I'm working with a retail inventory dataframe in pandas. Columns are
product_id, category, stock_level, last_restock_date (string),
sales_last_30_days, supplier_lead_time, and unit_cost.

I need to:
1. Calculate inventory turnover rate (sales_last_30_days / stock_level).
2. Flag slow-moving items (low turnover).
3. Predict potential stockouts by comparing days-of-stock-left to
   supplier_lead_time.
4. Plot a few charts (turnover distribution, stockout risk by category).

Please use pandas and matplotlib. Assume today is the reference date.
```

### Generated Code

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# tiny sample so the AI's code runs on its own
np.random.seed(0)
sample = pd.DataFrame({
    "product_id": [f"P{i:03d}" for i in range(1, 11)],
    "category": np.random.choice(["Electronics", "Grocery", "Apparel", "Home"], 10),
    "stock_level": np.random.randint(0, 200, 10),
    "last_restock_date": [(datetime.today() - timedelta(days=int(d))).strftime("%Y-%m-%d")
                          for d in np.random.randint(1, 120, 10)],
    "sales_last_30_days": np.random.randint(0, 50, 10),
    "supplier_lead_time": np.random.randint(2, 30, 10),
    "unit_cost": np.round(np.random.uniform(2, 80, 10), 2),
})

df = sample.copy()
df["last_restock_date"] = pd.to_datetime(df["last_restock_date"])
today = pd.Timestamp.today().normalize()

df["days_since_restock"] = (today - df["last_restock_date"]).dt.days
df["turnover_rate"] = df["sales_last_30_days"] / df["stock_level"].replace(0, np.nan)
df["days_of_stock_left"] = df["stock_level"] / (df["sales_last_30_days"] / 30).replace(0, np.nan)
df["stockout_risk"] = df["days_of_stock_left"] < df["supplier_lead_time"]
df["slow_mover"] = df["turnover_rate"] < 0.5

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["turnover_rate"].hist(ax=axes[0], bins=15)
axes[0].set_title("Turnover rate distribution")
df.groupby("category")["stockout_risk"].mean().plot(kind="bar", ax=axes[1])
axes[1].set_title("Stockout risk by category")
plt.tight_layout()
plt.show()

### Follow-up Prompts

- *If sales_last_30_days is 0 the days_of_stock_left will be inf, is that handled?*, the AI rephrased the divide to use `.replace(0, np.nan)` which I kept.
- *Can you add a quick category-level summary table?*, yes, added below.

### Final Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# sample data so the notebook runs
np.random.seed(0)
N = 60
products = pd.DataFrame({
    "product_id": [f"P{i:03d}" for i in range(1, N + 1)],
    "category": np.random.choice(["Electronics", "Grocery", "Apparel", "Home"], N),
    "stock_level": np.random.randint(0, 200, N),
    "last_restock_date": [
        (datetime.today() - timedelta(days=int(d))).strftime("%Y-%m-%d")
        for d in np.random.randint(1, 120, N)
    ],
    "sales_last_30_days": np.random.randint(0, 50, N),
    "supplier_lead_time": np.random.randint(2, 30, N),
    "unit_cost": np.round(np.random.uniform(2, 80, N), 2),
})

df = products.copy()
df["last_restock_date"] = pd.to_datetime(df["last_restock_date"])
today = pd.Timestamp.today().normalize()

# turnover, days of stock, stockout risk
df["days_since_restock"] = (today - df["last_restock_date"]).dt.days
df["turnover_rate"] = df["sales_last_30_days"] / df["stock_level"].replace(0, np.nan)
df["days_of_stock_left"] = df["stock_level"] / (df["sales_last_30_days"] / 30).replace(0, np.nan)
df["stockout_risk"] = df["days_of_stock_left"] < df["supplier_lead_time"]
df["slow_mover"] = df["turnover_rate"] < 0.5

df.head()

**Category summary**

In [ ]:
summary = df.groupby("category").agg(
    items=("product_id", "count"),
    avg_turnover=("turnover_rate", "mean"),
    slow_movers=("slow_mover", "sum"),
    stockout_risk=("stockout_risk", "sum"),
).round(3)
summary

**Visualizations**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["turnover_rate"].dropna(), bins=15, color="steelblue", edgecolor="white")
axes[0].set_title("Turnover rate distribution")
axes[0].set_xlabel("turnover (sales_30d / stock)")

df.groupby("category")["stockout_risk"].mean().plot(kind="bar", ax=axes[1], color="salmon")
axes[1].set_title("Stockout risk by category")
axes[1].set_ylabel("share of items at risk")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### Discussion - Scenario 1

- **Date calculations**: the AI assumed `last_restock_date` was a string and used `pd.to_datetime` first, then computed days against `pd.Timestamp.today().normalize()`. I kept that. `normalize()` matters because otherwise the diffs shift around through the day.
- **Visualization approaches**: first pass just had a histogram. I asked for a second view by category and the AI suggested a bar of the stockout-rate mean. That one is more useful for the manager than the raw histogram.
- **Error handling**: I added `.replace(0, np.nan)` on both denominators so a zero stock or zero sales doesn't blow up into `inf`. The AI's first answer missed this and I had to ask a follow-up.

### AI Integration Strategy - Scenario 1

**1. Date handling / `normalize()`**

- **Prompt**: *For days_since_restock I'm using today - last_restock_date. Should I worry about the time component making the diff jump by 1?*
- **AI response**: *Yes , use pd.Timestamp.today().normalize() so both sides are at midnight. Otherwise the diff can be off by one as the day rolls over.
- **Implementation**: kept `.normalize()` on the today line.
- **Modifications**: none , that was genuinely helpful, I wouldn't have thought of it.

**2. Division-by-zero in turnover / days-of-stock**

- **Prompt**: *If sales_last_30_days is 0 the days_of_stock_left will be inf , is that handled?*
- **AI response**: *Use .replace(0, np.nan) on the denominator so you get NaN instead of inf, which is easier to plot and groupby.*
- **Implementation**: applied to both `stock_level` and `sales_last_30_days / 30` denominators.
- **Modifications**: I extended it to both denominators since the AI only named one. The stock=0 case was the same problem.

---

## Scenario 2: Website Analytics Debug

### Setup

Marketing says the engagement metrics look wrong , bounce rates over 100% and negative session times. The original code is:

```python
def analyze_user_engagement(logs_df):
    metrics = {
        'bounce_rate': logs_df.groupby('session_id')['page_views'].apply(
            lambda x: x == 1).mean(),
        'avg_session_time': logs_df.groupby('session_id')['duration'].sum(),
        'pages_per_session': logs_df.groupby('session_id')['page_views'].mean()
    }

    device_metrics = logs_df.groupby('device_type').agg({
        'session_id': 'count',
        'duration': 'mean',
        'page_views': 'sum'
    })

    return metrics, device_metrics
```

### My Prompt

```
This pandas function calculates website session metrics. Two problems
the team is reporting: bounce rate comes back over 100%, and session
times are sometimes negative.

Please:
1. Identify the bugs in the current code.
2. Fix the bounce rate calculation so it's a 0..1 ratio.
3. Make sure session duration is non-negative and is summed per session
   (not averaged across rows).
4. Add basic data validation (negative durations, missing values).
5. Show a chart of bounce rate by device_type.

Use pandas and matplotlib.
```

### Generated Code

In [ ]:
def analyze_user_engagement(logs_df):
    # validate
    if (logs_df["duration"] < 0).any():
        raise ValueError("duration has negative values")
    if logs_df["session_id"].isna().any():
        raise ValueError("session_id has nulls")

    sessions = logs_df.groupby("session_id").agg(
        page_views=("page_views", "sum"),
        duration=("duration", "sum"),
        device_type=("device_type", "first"),
    ).reset_index()

    sessions["bounce"] = sessions["page_views"] == 1
    metrics = {
        "bounce_rate": sessions["bounce"].mean(),          # 0..1
        "avg_session_time": sessions["duration"].mean(),   # per session
        "pages_per_session": sessions["page_views"].mean(),
    }

    device_metrics = logs_df.groupby("device_type").agg(
        sessions=("session_id", "nunique"),
        avg_duration=("duration", "mean"),
        page_views=("page_views", "sum"),
    )

    return metrics, device_metrics

### Follow-up Prompts

- *In the original code, `page_views` wasn't summed per session, same with `duration`. The original did `.sum()` on duration but `.mean()` was on raw rows. Should I sum per session?*, yes, this is what the original was getting wrong.
- *For the device-level metric, do I want nunique sessions or row count?*, nunique, otherwise tablet rows are over-counted.

### Final Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# sample logs (one row per page view)
logs = pd.DataFrame({
    "session_id": [1, 1, 1, 2, 2, 3, 4, 4, 5, 5, 5, 6],
    "page_views": [3, 3, 3, 1, 1, 2, 4, 4, 1, 1, 1, 5],
    "duration":   [300, 300, 300, 60, 60, 180, 400, 400, 30, 30, 30, 500],
    "device_type": ["mobile", "mobile", "mobile", "desktop", "desktop",
                    "mobile", "tablet", "tablet", "desktop", "desktop",
                    "desktop", "mobile"],
})

logs.head()

In [ ]:
def analyze_user_engagement(logs_df):
    # validation
    if logs_df["session_id"].isna().any():
        raise ValueError("session_id has nulls")
    if (logs_df["duration"] < 0).any():
        raise ValueError("duration has negative values")
    if (logs_df["page_views"] < 0).any():
        raise ValueError("page_views has negative values")

    # collapse to one row per session
    sessions = (
        logs_df.groupby("session_id")
        .agg(page_views=("page_views", "sum"),
             duration=("duration", "sum"),
             device_type=("device_type", "first"))
        .reset_index()
    )

    sessions["bounce"] = sessions["page_views"] == 1
    metrics = {
        # bounce rate is a 0..1 ratio , not a percent
        "bounce_rate": sessions["bounce"].mean(),
        "avg_session_time_sec": sessions["duration"].mean(),
        "pages_per_session": sessions["page_views"].mean(),
    }

    device_metrics = logs_df.groupby("device_type").agg(
        sessions=("session_id", "nunique"),
        avg_duration=("duration", "mean"),
        page_views=("page_views", "sum"),
    ).reset_index()

    return metrics, device_metrics, sessions

metrics, device_metrics, sessions = analyze_user_engagement(logs)
metrics

**Device-level summary**

In [ ]:
device_metrics

**Bounce rate by device**

In [ ]:
device_bounce = (
    sessions.groupby("device_type")["bounce"].mean().reindex(
        device_metrics["device_type"]
    )
)

fig, ax = plt.subplots(figsize=(6, 4))
device_bounce.plot(kind="bar", ax=ax, color="teal", edgecolor="black")
ax.set_title("Bounce rate by device type")
ax.set_ylabel("bounce rate (0..1)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Discussion - Scenario 2

- **Error identification**: the AI flagged two things right away , the original `bounce_rate` was applied to a `groupby` that already aggregated `page_views`, so the `x == 1` check was comparing sums to 1, and the mean was over raw page-view rows, not sessions. The over 100% in the original comes from `mean()` running across rows, not sessions.
- **Validation**: I asked for explicit checks on negative durations and null `session_id`. The AI returned a `raise ValueError` pattern which I kept.
- **Time handling**: the original `sum()` on duration was correct per session, but only if the source data is one row per session. The fix above assumes one row per page view (so we need to sum per session). The negative times weren't actually in the data , the report was probably confusion from the comparison-not-a-percent bug.

### AI Integration Strategy - Scenario 2

**1. Why the original bounce rate was over 100%**

- **Prompt**: *Why would the original code return a bounce rate over 100%?*
- **AI response**: *Because the .mean() is computed across page-view rows, not sessions. If a session has 3 page views, all 3 rows are counted toward the average. The bounce_rate is a per-session metric, so the groupby+agg needs to happen first, then mean() over the boolean.*
- **Implementation**: collapsed to one row per session first, then `sessions["bounce"].mean()`.
- **Modifications**: none , diagnosis was correct.

**2. `nunique` vs `count` for device sessions**

- **Prompt**: *For device_metrics, should session_id be count or nunique?*
- **AI response**: *nunique , count counts page-view rows, which biases toward devices that have more page views per session.*
- **Implementation**: switched to `nunique`.
- **Modifications**: none.

---

## Scenario 3: Customer Segmentation Query

### Setup

Product team needs to find active, high-value customers and look at their preferences before a feature rollout.

Tables:
- `user_activity(user_id, last_login_date, feature_usage_count, account_type)`
- `transactions(transaction_id, user_id, transaction_date, amount, platform)`
- `user_preferences(user_id, communication_preference, interface_theme, notification_settings)`

Goal: active users (logged in last 30 days), top 20% by spending, plus their preference trends.

### My Prompt

```
Write a SQL query that pulls:
- Active users (last_login_date within the last 30 days).
- Filtered to the top 20% by total spend.
- Joined to user_preferences to see communication_preference,
  interface_theme, and notification_settings.

Schema:
  user_activity(user_id, last_login_date, feature_usage_count, account_type)
  transactions(transaction_id, user_id, transaction_date, amount, platform)
  user_preferences(user_id, communication_preference, interface_theme, notification_settings)

Use a CTE. Show me the spend per user, the percentile cutoff, and the
final joined result ordered by total spend DESC.
```

### Generated Code

In [ ]:
# SQL the AI generated (Postgres-style, commented out so this notebook cell stays Python):
#
# WITH user_spend AS (
#     SELECT user_id, SUM(amount) AS total_spend
#     FROM transactions
#     GROUP BY user_id
# ),
# ranked AS (
#     SELECT user_id, total_spend,
#            PERCENT_RANK() OVER (ORDER BY total_spend DESC) AS pct_rank
#     FROM user_spend
# )
# SELECT u.user_id, r.total_spend, u.last_login_date, u.feature_usage_count,
#        u.account_type, p.communication_preference, p.interface_theme,
#        p.notification_settings
# FROM user_activity u
# JOIN ranked r ON u.user_id = r.user_id
# LEFT JOIN user_preferences p ON u.user_id = p.user_id
# WHERE u.last_login_date >= DATE('now', '-30 days')
#   AND r.pct_rank <= 0.20
# ORDER BY r.total_spend DESC;

print("see the SQL in the comments above")

### Follow-up Prompts

- *Should I use PERCENT_RANK or NTILE(5) for the top 20%?*, AI suggested PERCENT_RANK because it gives an exact 20% cutoff regardless of how many users there are. NTILE puts you in a bucket regardless of values.
- *Is the date filter better as DATE('now', '-30 days') or computed in Python first?*, both work, keeping it in SQL.

### Final Solution

In [ ]:
import sqlite3, pandas as pd

con = sqlite3.connect(":memory:")
con.execute("""CREATE TABLE user_activity (
    user_id INTEGER, last_login_date TEXT,
    feature_usage_count INTEGER, account_type TEXT)""")
con.execute("""CREATE TABLE transactions (
    transaction_id INTEGER PRIMARY KEY, user_id INTEGER,
    transaction_date TEXT, amount REAL, platform TEXT)""")
con.execute("""CREATE TABLE user_preferences (
    user_id INTEGER, communication_preference TEXT,
    interface_theme TEXT, notification_settings TEXT)""")

con.executemany("INSERT INTO user_activity VALUES (?,?,?,?)", [
    (1, "2026-07-20", 12, "premium"),
    (2, "2026-06-01",  3, "free"),     # not active
    (3, "2026-07-22", 27, "premium"),
    (4, "2026-07-19",  8, "free"),
])
con.executemany("INSERT INTO transactions VALUES (?,?,?,?,?)", [
    (1, 1, "2026-07-01", 250.0, "web"),
    (2, 1, "2026-07-10", 150.0, "ios"),
    (3, 2, "2026-05-01",  10.0, "web"),
    (4, 3, "2026-07-05", 800.0, "android"),
    (5, 3, "2026-07-15", 200.0, "ios"),
    (6, 4, "2026-07-18",  90.0, "web"),
    (7, 5, "2026-07-21", 999.0, "ios"),  # active but no user_activity row
])
con.executemany("INSERT INTO user_preferences VALUES (?,?,?,?)", [
    (1, "email", "dark", "all"),
    (3, "sms",   "light", "digest"),
    (4, "email", "dark", "all"),
])

sql = """
WITH user_spend AS (
    SELECT user_id, SUM(amount) AS total_spend
    FROM transactions
    GROUP BY user_id
),
ranked AS (
    SELECT user_id, total_spend,
           PERCENT_RANK() OVER (ORDER BY total_spend DESC) AS pct_rank
    FROM user_spend
)
SELECT u.user_id, r.total_spend, u.last_login_date, u.feature_usage_count,
       u.account_type, p.communication_preference, p.interface_theme,
       p.notification_settings
FROM user_activity u
JOIN ranked r ON u.user_id = r.user_id
LEFT JOIN user_preferences p ON u.user_id = p.user_id
WHERE u.last_login_date >= DATE('now', '-30 days')
  AND r.pct_rank <= 0.20
ORDER BY r.total_spend DESC;
"""

df = pd.read_sql_query(sql, con)
df

**Preference trend for the high-value segment**

In [ ]:
if len(df):
    print("communication_preference:")
    print(df["communication_preference"].value_counts(normalize=True).round(2))
    print()
    print("interface_theme:")
    print(df["interface_theme"].value_counts(normalize=True).round(2))
    print()
    print("account_type:")
    print(df["account_type"].value_counts(normalize=True).round(2))
else:
    print("no high-value customers in the seed data")

### Discussion - Scenario 3

- **Percentile calculations**: I used `PERCENT_RANK() OVER (ORDER BY total_spend DESC)` and filtered `pct_rank <= 0.20`. The AI's first answer used `NTILE(5)` and I asked about the difference. `PERCENT_RANK` is the right pick because the cutoff is exactly 20% of users regardless of how many there are; `NTILE(5)` just puts you in a bucket.
- **Date filtering**: `DATE('now', '-30 days')` keeps the cutoff in SQL instead of hardcoding a date from Python. SQLite supports it; in Postgres you'd use `CURRENT_DATE - INTERVAL '30 days'`.
- **Query structure**: CTE with `user_spend` → `ranked` → final join. Doing the spend aggregation and the window in CTEs keeps the outer query readable and lets the optimizer reuse the spend calc if the query is reused. I used a LEFT JOIN to `user_preferences` so a user without a preferences row still shows up.

### AI Integration Strategy - Scenario 3

**1. PERCENT_RANK vs NTILE for top 20%**

- **Prompt**: *Should I use PERCENT_RANK() or NTILE(5) to get the top 20% by spend?*
- **AI response**: *PERCENT_RANK() gives an exact 20% cutoff regardless of group size. NTILE(5) puts you in a bucket, so you can end up with 19% or 22% depending on rounding.*
- **Implementation**: used `PERCENT_RANK() OVER (ORDER BY total_spend DESC)` and `pct_rank <= 0.20`.
- **Modifications**: none , the explanation was clear and the result matches what I wanted.

**2. CTE vs subquery for the spend aggregation**

- **Prompt**: *Should I put the spend aggregation in a CTE or a subquery in the WHERE clause?*
- **AI response**: *CTE , it's clearer and Postgres' optimizer handles them similarly to subqueries for this kind of aggregation.*
- **Implementation**: kept `user_spend` and `ranked` as separate CTEs.
- **Modifications**: none.

---

## Final Thoughts

Across the three scenarios, the AI was most useful for:
- Catching a subtle bug (the over-100% bounce rate).
- Suggesting `.normalize()` and the division-by-zero guard.
- Explaining PERCENT_RANK vs NTILE.

It was less useful for choosing which columns to create or which charts to draw , those decisions came from the actual problem framing (what a store manager or marketer would want to see).